# 04 · Visual Analytics
**Flight Analytics Platform**

Advanced interactive visualisations using Plotly:
- World flight density map
- Flight traffic over time
- Velocity vs altitude heatmap
- Country leaderboard
- Real-time 3D flight globe
- Flight region radar

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import psycopg2
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DARK = 'plotly_dark'
ACC  = '#00d4ff'

def get_conn():
    return psycopg2.connect(
        dbname  = os.getenv('POSTGRES_DB',       'flightdw'),
        user    = os.getenv('POSTGRES_USER',     'flightuser'),
        password= os.getenv('POSTGRES_PASSWORD', 'flightpass123'),
        host    = os.getenv('POSTGRES_HOST',     'postgres'),
        port    = os.getenv('POSTGRES_PORT',     '5432'),
    )

def sql(q): 
    with get_conn() as c: return pd.read_sql(q, c)

print('✅ Ready')

## 1. World Flight Density Map

In [ ]:
flights = sql("""
    SELECT longitude, latitude, avg_velocity_kmh, callsign, origin_country, altitude_level
    FROM main.flight_analytics
    WHERE longitude IS NOT NULL AND latitude IS NOT NULL
      AND last_updated >= NOW() - INTERVAL '30 minutes'
    LIMIT 3000
""")

print(f'Flights loaded: {len(flights):,}')

if flights.empty:
    print('⚠️  No data yet — seeding synthetic positions for demo')
    rng = np.random.default_rng(42)
    n = 2000
    flights = pd.DataFrame({
        'longitude':       rng.uniform(-180, 180, n),
        'latitude':        rng.uniform(-60, 80, n),
        'avg_velocity_kmh':rng.uniform(200, 900, n),
        'callsign':        [f'FLT{i:04d}' for i in range(n)],
        'origin_country':  rng.choice(['Germany','USA','UK','France','Japan','China'], n),
        'altitude_level':  rng.choice(['medium','high','very_high'], n),
    })

fig = go.Figure(go.Densitymapbox(
    lat=flights['latitude'],
    lon=flights['longitude'],
    z=flights['avg_velocity_kmh'].fillna(500),
    radius=10,
    colorscale='YlOrRd',
    opacity=0.75,
))
fig.update_layout(
    title='🌏 World Flight Density Heatmap',
    mapbox_style='carto-darkmatter',
    mapbox=dict(center=dict(lat=30, lon=10), zoom=1.2),
    height=480,
    template=DARK,
    margin=dict(l=0,r=0,t=40,b=0),
)
fig.show()

## 2. Flight Traffic Over Time

In [ ]:
ts_df = sql("""
    SELECT DATE_TRUNC('hour', bucket) AS hour,
           SUM(flight_count)   AS flights,
           AVG(avg_velocity)   AS avg_speed,
           AVG(avg_altitude)   AS avg_altitude
    FROM main.hourly_flight_counts
    WHERE bucket >= NOW() - INTERVAL '48 hours'
    GROUP BY 1
    ORDER BY 1
""")

if ts_df.empty:
    # Synthetic time series
    hours = pd.date_range(end=pd.Timestamp.now(), periods=48, freq='h')
    rng = np.random.default_rng(42)
    ts_df = pd.DataFrame({
        'hour':        hours,
        'flights':     rng.integers(5000, 15000, 48),
        'avg_speed':   rng.uniform(600, 850, 48),
        'avg_altitude':rng.uniform(7000, 11000, 48),
    })

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Active Flights per Hour', 'Avg Speed (km/h)'],
                    row_heights=[0.6, 0.4])

fig.add_trace(go.Scatter(
    x=ts_df['hour'], y=ts_df['flights'],
    fill='tozeroy', fillcolor='rgba(0,212,255,0.1)',
    line=dict(color=ACC, width=2), name='Flights'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=ts_df['hour'], y=ts_df['avg_speed'],
    line=dict(color='#f97316', width=1.5, dash='dot'), name='Avg Speed'
), row=2, col=1)

fig.update_layout(
    title='📈 Flight Traffic Time Series (48h)',
    template=DARK, height=420,
    legend=dict(orientation='h', y=1.12),
)
fig.show()

## 3. Velocity vs Altitude 2D Density Heatmap

In [ ]:
va = sql("""
    SELECT avg_velocity_kmh, avg_altitude
    FROM main.flight_analytics
    WHERE avg_velocity_kmh > 50
      AND avg_altitude > 100
    LIMIT 5000
""")

if va.empty:
    rng = np.random.default_rng(42)
    n = 3000
    alt = rng.uniform(1000, 12000, n)
    va = pd.DataFrame({
        'avg_velocity_kmh': 200 + alt*0.04 + rng.normal(0,60,n),
        'avg_altitude':     alt,
    })

fig = px.density_heatmap(
    va, x='avg_velocity_kmh', y='avg_altitude',
    nbinsx=40, nbinsy=40,
    color_continuous_scale='Viridis',
    title='🔥 Velocity vs Altitude 2D Density',
    labels={'avg_velocity_kmh':'Speed (km/h)','avg_altitude':'Altitude (m)'},
    template=DARK,
)
fig.update_layout(height=420)
fig.show()

## 4. Country Leaderboard — Animated Bar Race Snapshot

In [ ]:
country_df = sql("""
    SELECT origin_country,
           SUM(flight_count) AS flights,
           AVG(avg_altitude) AS avg_alt,
           AVG(avg_velocity) AS avg_speed
    FROM main.country_traffic_summary
    WHERE origin_country IS NOT NULL
      AND snapshot_time >= NOW() - INTERVAL '1 hour'
    GROUP BY origin_country
    ORDER BY flights DESC
    LIMIT 20
""")

if country_df.empty:
    rng = np.random.default_rng(42)
    countries = ['United States','Germany','China','France','United Kingdom',
                 'Japan','Turkey','Spain','Australia','Canada','Brazil',
                 'Russia','India','Netherlands','Italy','South Korea',
                 'Switzerland','Austria','Poland','Norway']
    country_df = pd.DataFrame({
        'origin_country': countries,
        'flights':        rng.integers(200, 4000, len(countries)),
        'avg_alt':        rng.uniform(7000, 11000, len(countries)),
        'avg_speed':      rng.uniform(650, 850, len(countries)),
    }).sort_values('flights', ascending=False)

country_df = country_df.sort_values('flights')

fig = go.Figure(go.Bar(
    x=country_df['flights'],
    y=country_df['origin_country'],
    orientation='h',
    marker=dict(
        color=country_df['flights'],
        colorscale='Plasma',
        line=dict(color='rgba(0,0,0,0.2)', width=0.5),
    ),
    text=country_df['flights'].apply(lambda x: f'{x:,}'),
    textposition='outside',
))
fig.update_layout(
    title='🏆 Top Countries by Active Flights',
    xaxis_title='Active Flights', yaxis_title=None,
    template=DARK, height=500,
    showlegend=False, coloraxis_showscale=False,
)
fig.show()

## 5. 3D Globe — Flight Arcs

In [ ]:
map_df = sql("""
    SELECT callsign, longitude, latitude, avg_velocity_kmh, altitude_level
    FROM main.flight_analytics
    WHERE longitude IS NOT NULL AND latitude IS NOT NULL
      AND last_updated >= NOW() - INTERVAL '30 minutes'
    LIMIT 1500
""")

if map_df.empty:
    map_df = flights.copy()

alt_color = {'ground':'#ef4444','low':'#f97316','medium':'#eab308',
             'high':'#22c55e','very_high':'#3b82f6'}

fig = px.scatter_geo(
    map_df,
    lat='latitude', lon='longitude',
    color='altitude_level',
    color_discrete_map=alt_color,
    hover_name='callsign',
    size_max=4,
    title='🌐 3D Globe — Live Flight Positions',
    template=DARK,
    projection='orthographic',
)
fig.update_geos(
    showcoastlines=True, coastlinecolor='#444',
    showland=True,  landcolor='#1a1a2e',
    showocean=True, oceancolor='#050510',
    projection_rotation=dict(lon=10, lat=20),
)
fig.update_layout(height=520, legend_title='Altitude Level')
fig.show()

## 6. Flight Region Radar Chart

In [ ]:
region_df = sql("""
    SELECT flight_region,
           COUNT(DISTINCT icao24) AS flights,
           AVG(avg_velocity_kmh)  AS avg_speed,
           AVG(avg_altitude)      AS avg_alt
    FROM main.flight_analytics
    WHERE flight_region NOT IN ('unknown')
      AND last_updated >= NOW() - INTERVAL '30 minutes'
    GROUP BY flight_region
""")

if region_df.empty:
    region_df = pd.DataFrame({
        'flight_region': ['europe_africa','north_america','asia_pacific',
                          'south_america','arctic','other'],
        'flights':  [4500, 3200, 2800, 600, 150, 400],
        'avg_speed':[740, 780, 720, 680, 550, 600],
        'avg_alt':  [10200, 10500, 9800, 8700, 8000, 9000],
    })

cats = list(region_df['flight_region'])

fig = go.Figure()
for metric, color, name in [
    ('flights',  ACC,       'Flight Count'),
    ('avg_speed','#f97316', 'Avg Speed (km/h)'),
]:
    vals = region_df[metric].tolist()
    vals_norm = (np.array(vals) / max(vals) * 100).tolist()  # normalise 0–100
    fig.add_trace(go.Scatterpolar(
        r=vals_norm + [vals_norm[0]],
        theta=cats + [cats[0]],
        fill='toself',
        fillcolor=f'rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.15)',
        line=dict(color=color, width=2),
        name=name,
    ))

fig.update_layout(
    title='🗺️ Flight Activity by Region (normalised)',
    polar=dict(
        bgcolor='#0d1117',
        radialaxis=dict(visible=True, range=[0, 110], gridcolor='#333'),
        angularaxis=dict(gridcolor='#333'),
    ),
    template=DARK,
    height=420,
    legend=dict(orientation='h', y=-0.12),
)
fig.show()